In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from IPython.display import display
import urllib.request
import warnings
warnings.filterwarnings("ignore")

## Khởi tạo Julia và tải dữ liệu benchmark

In [ ]:
!curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz -o julia.tar.gz
!tar -xzf julia.tar.gz -C /usr --strip-components 1

In [ ]:
!git clone https://github.com/cosude/Data-Mining-FIM-Group-7.git
%cd Data-Mining-FIM-Group-7
!julia --project=. -e 'using Pkg; Pkg.instantiate()'
!julia --project=. download_benchmark.jl

## Chương 1: Cài đặt và kiểm thử

In [ ]:
!julia --project=. tests/runtests.jl

Kết quả unit test cho thấy hai cài đặt cơ bản và tối ưu của giải thuật FP-growth khai thác được các tập phổ biến với support count khớp với phần mềm SPMF trên $5$ cơ sở dữ liệu gồm: chess, mushroom, accidents, retail và T1014D100K.

Ngoài ra, cài đặt tối ưu có thời gian chạy nhanh hơn khoảng $1.7$ lần và tiết kiệm bộ nhớ khoảng $2.8$ lần so với cài đặt cơ bản.

## Chương 2: Thực nghiệm và đánh giá

**Lưu ý: các ô markdown nhận xét kết quả thực nghiệm chỉ mang tính tổng quát, các phân tích chi tiết sẽ được trình bày cụ thể trong báo cáo để tránh notebook demo quá dài.**

### 2.1. Kết quả đối chiếu với SPMF

In [ ]:
!julia --project=. experiments/exp1_correctness.jl

Kết quả trên cho thấy hai cài đặt của nhóm đạt tỉ lệ khớp $100.0\%$ trên toàn bộ các bộ dữ liệu thử nghiệm, giá trị support count của từng itemset cũng khớp với SPMF.

### 2.2. Mối quan hệ giữa minsupport và thời gian chạy

In [ ]:
!julia --project=. experiments/exp2_runtime_vs_minsup.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

csv_path = "data/results/exp2_runtime_vs_minsup/exp2_runtime.csv"
out_dir = "data/results/exp2_runtime_vs_minsup"
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(csv_path)
datasets = df['Dataset'].unique()

for dataset in datasets:
    data = df[df['Dataset'] == dataset]
    
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(data['Minsup'], data['SPMF_Time_s'], 
             label="SPMF (Java)", marker='o', lw=2, color='blue', linestyle='-')
             
    plt.plot(data['Minsup'], data['Base_Time_s'], 
             label="Basic (Julia)", marker='s', lw=2, color='red', linestyle='--')
             
    plt.plot(data['Minsup'], data['Opt_Time_s'], 
             label="Optimized (Julia)", marker='D', lw=2, color='green', linestyle=':')
    
    plt.gca().invert_xaxis()
    
    plt.title(f"Runtime vs. Minsup ({dataset.capitalize()})")
    plt.xlabel("Minimum Support")
    plt.ylabel("Runtime (Seconds)")
    
    plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
    
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    plt.savefig(os.path.join(out_dir, f"plot_runtime_{dataset}.png"), bbox_inches="tight")
    plt.show()

Có thể thấy xu hướng của cả 3 đường (SPMF, cơ bản, tối ưu) đều đi lên trên cả 5 đồ thị. Điều này cho thấy thời gian chạy của thuật toán FP-growth tăng dần khi minsupport giảm.

Mối quan hệ này có thể được lí giải dựa vào ý nghĩa của minsupport đối với bài toán khai thác tập phổ biến. Khi minsupport nhỏ đi, số lượng các itemsets - đặc biệt là 1-itemsets - phổ biến tăng lên, làm cho FP-tree dày đặc hơn. Vì thuật toán FP-growth khai thác trên chính FP-tree, chi phí tính toán cũng đồng thời tăng dẫn tới thời gian thực thi tăng.

### 2.3. Mối quan hệ giữa số lượng tập phổ biến và minsupport

In [ ]:
!julia --project=. experiments/exp3_itemset_count.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

csv_path = "data/results/exp3_itemset_count/exp3_itemset_counts.csv"
out_dir = "data/results/exp3_itemset_count"
os.makedirs(out_dir, exist_ok=True)

df_counts = pd.read_csv(csv_path)
datasets = df_counts['Dataset'].unique()

for dataset in datasets:
    data = df_counts[df_counts['Dataset'] == dataset]
    ds_type = data['Type'].iloc[0]
    
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(data['Minsup'], data['Itemset_Count'], 
             label="Itemset Count", marker='o', lw=2.5, color='blue', linestyle='-')
    
    plt.gca().invert_xaxis()
    
    plt.yscale('log')
    
    plt.title(f"Output Size vs. Minsup\n({dataset.capitalize()} - {ds_type})")
    plt.xlabel("Minimum Support")
    plt.ylabel("Number of Itemsets (Log Scale)")
    
    plt.grid(True, which='major', alpha=0.5)
    plt.grid(True, which='minor', alpha=0.2, linestyle=':')
    
    plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
    plt.tight_layout()
    
    plt.savefig(os.path.join(out_dir, f"plot_count_{dataset}.png"), bbox_inches="tight")
    plt.show()

Quan sát trên cả 5 đồ thị, khi minsupport giảm dần, số lượng tập phổ biến khai thác được tăng lên theo cấp số nhân.

Đối với các tập dữ liệu dày đặc, khi các giao dịch dài và các item lặp lại nhiều, số lượng tập phổ biến được khai thác rất nhiều và tương đối nhạy cảm với minsupport.

Ngược lại, các tập dữ liệu thưa có các giao dịch tương đối ngắn, các item lặp lại ít, số lượng tập phổ biến ít hơn nhiều so với các tập dữ liệu dày đặc ở cùng một ngưỡng minsupport và thay đổi ít hơn khi minsupport thay đổi.

### 2.4. So sánh hiệu năng bộ nhớ

In [ ]:
!julia --project=. experiments/exp4_memory.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

dir_path = "data/results/exp4_memory"
csv_path = os.path.join(dir_path, "exp4_memory.csv")
os.makedirs(dir_path, exist_ok=True)

df = pd.read_csv(csv_path)

base = df[df['Impl'] == 'base'].copy().reset_index(drop=True)
opt = df[df['Impl'] == 'opt'].copy().reset_index(drop=True)

names = base['Dataset'].values
x = np.arange(len(names))
width = 0.4

metrics = [
    ('Peak_RSS_MB', 'Peak RAM (whole process)', 'plot_peak_rss.png'),
    ('Delta_RSS_MB', 'Extra peak RAM used by mining (Δ RSS)', 'plot_delta_rss.png'),
    ('Alloc_MB', 'Total allocated memory (GC pressure)', 'plot_alloc.png')
]

for col, ttl, fname in metrics:
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.bar(x - width/2, base[col], width=width, label="Basic implementation", color='#d62728', alpha=0.8)
    plt.bar(x + width/2, opt[col], width=width, label="Optimized implementation", color='#2ca02c', alpha=0.8)
    
    plt.xticks(x, names)
    plt.title(ttl)
    plt.ylabel("MB")
    plt.legend(loc="upper left")
    plt.grid(True, axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(dir_path, fname), bbox_inches="tight")
    plt.show()

base['Tree_Nodes_Safe'] = base['Tree_Nodes'].apply(lambda n: max(n, 1))
bytes_per_node = (base['Tree_MB'] * (1024**2)) / base['Tree_Nodes_Safe']

plt.figure(figsize=(9, 5), dpi=300)
plt.bar(x, bytes_per_node, width=0.6, label="Basic FP-tree", color='#1f77b4', alpha=0.8)

plt.xticks(x, names)
plt.title("Memory per FP-tree node")
plt.ylabel("Bytes / node")
plt.legend(loc="upper right")
plt.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_bytes_per_node.png"), bbox_inches="tight")
plt.show()

print("\n" + "="*80)
print("BẢNG TÓM TẮT THÔNG SỐ (BASE IMPLEMENTATION)")
print("="*80)
base_summary = base[['Dataset', 'Minsup', 'Tree_Nodes', 'Tree_MB', 'DB_MB', 'Peak_RSS_MB', 'Delta_RSS_MB']]
display(base_summary)

print("\n" + "="*80)
print("BẢNG THAM KHẢO SPMF (JVM HEAP)")
print("="*80)
spmf = df[df['Impl'] == 'spmf'].dropna(axis=1, how='all')
display(spmf)

Các số liệu và biểu đồ từ trên cho thấy các kỹ thuật tối ưu bộ nhớ ở như khử đệ quy, sử dụng BitArray cho quá trình xây FP-tree cũng như sử dụng single-prefix path cho quá trình khai thác tập phổ biến mang lại hiệu quả về bộ nhớ đáng kể cho cài đặt tối ưu so với cơ bản.

Mặc dù số nút và kích thước của FP-tree tương đương nhau, số lượng bộ nhớ cấp phát của bản tối ưu ít hơn nhiều so với cơ bản, thể hiện rõ trên $5$ tập dữ liệu. Sự khác biệt này có thể được giải thích chủ yếu nhờ vào việc sử dụng BitArray chia sẻ chung cho các giao dịch, giảm đáng kể lượng bộ nhớ cần cấp phát riêng.

### 2.5. Khả năng mở rộng đối với các mức kích thước cơ sở dữ liệu

In [ ]:
!julia --project=. experiments/exp5_scalability.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

dir_path = "data/results/exp5_scalability"
csv_path = os.path.join(dir_path, "exp5_scalability.csv")
os.makedirs(dir_path, exist_ok=True)

df = pd.read_csv(csv_path)

def linear_fit(x, y):
    x_mean, y_mean = np.mean(x), np.mean(y)
    a = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
    b = y_mean - a * x_mean
    
    y_pred = a * x + b
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y_mean)**2)
    r2 = 1 - (ss_res / ss_tot)
    
    return a, b, r2

datasets = df['Dataset'].unique()

for dataset in datasets:
    data = df[df['Dataset'] == dataset].copy()
    
    n = data['Transactions'] / 1000
    minsup_val = data['Minsup'].iloc[0]
    
    a, b, r2 = linear_fit(n, data['Base_Time_s'])
    
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(n, data['Base_Time_s'], label="Basic implementation", marker='s', lw=2, color='#d62728')
    plt.plot(n, data['Opt_Time_s'], label="Optimized implementation", marker='D', lw=2, color='#2ca02c')
    
    if not data['SPMF_Time_s'].isna().all():
        plt.plot(n, data['SPMF_Time_s'], label="SPMF", marker='o', lw=2, color='#1f77b4')
        
    plt.plot(n, data['Base_Build_s'], label="Basic: FP-tree construction only", marker='^', ls=':', color='purple')
    plt.plot(n, a * n + b, label=f"Linear fit (R² = {r2:.3f})", ls='--', color='gray')
    
    plt.title(f"Scalability ({dataset.capitalize()}, minsup = {minsup_val})")
    plt.xlabel("Number of transactions (K)")
    plt.ylabel("Runtime (Seconds)")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(dir_path, f"plot_scalability_{dataset}.png"), bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(n, data['Base_Time_s'] / n, label="Basic", marker='s', lw=2, color='#d62728')
    plt.plot(n, data['Opt_Time_s'] / n, label="Optimized", marker='D', lw=2, color='#2ca02c')
    
    plt.title(f"Runtime per 1K transactions ({dataset.capitalize()})")
    plt.xlabel("Number of transactions (K)")
    plt.ylabel("Seconds / 1K transactions")
    
    plt.ylim(bottom=0)
    
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(dir_path, f"plot_time_per_trans_{dataset}.png"), bbox_inches="tight")
    plt.show()

print("All Scalability plots (Experiment 5) have been saved successfully!")

### 2.6. Ảnh hưởng của độ dài trung bình của giao dịch

In [ ]:
!julia --project=. experiments/exp6_avg_length.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

dir_path = "data/results/exp6_avg_length"
csv_path = os.path.join(dir_path, "exp6_avg_length.csv")
os.makedirs(dir_path, exist_ok=True)

df = pd.read_csv(csv_path)

T = df['Actual_Avg_Len']
minsup_val = df['Minsup'].iloc[0]

plt.figure(figsize=(9, 5), dpi=300)

plt.plot(T, df['Base_Time_s'], label="Basic implementation", marker='s', lw=2, color='#d62728', linestyle='--')
plt.plot(T, df['Opt_Time_s'], label="Optimized implementation", marker='D', lw=2, color='#2ca02c', linestyle=':')

if not df['SPMF_Time_s'].isna().all():
    plt.plot(T, df['SPMF_Time_s'], label="SPMF", marker='o', lw=2, color='#1f77b4', linestyle='-')

plt.plot(T, df['Base_Build_s'], label="Basic: FP-tree construction only", marker='^', lw=2, color='purple', linestyle=':')

plt.title(f"Runtime vs. Avg. Length (minsup = {minsup_val})")
plt.xlabel("Average transaction length")
plt.ylabel("Runtime (Seconds)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_runtime_vs_length.png"), bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5), dpi=300)

plt.plot(T, df['Itemsets'], label="Frequent itemsets", marker='o', lw=2, color='blue', linestyle='-')
plt.plot(T, df['Tree_Nodes'], label="FP-tree nodes", marker='s', lw=2, color='#ff7f0e', linestyle='--')

plt.title("Itemsets and tree size vs. Avg. Length")
plt.xlabel("Average transaction length")
plt.ylabel("Count (log scale)")
plt.yscale('log')

plt.legend(loc="lower right")
plt.grid(True, which='major', alpha=0.5)
plt.grid(True, which='minor', alpha=0.2, linestyle=':')

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_count_vs_length.png"), bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5), dpi=300)

plt.plot(df['Itemsets'], df['Base_Time_s'], label="Basic", marker='s', lw=2, color='#d62728', linestyle='--')
plt.plot(df['Itemsets'], df['Opt_Time_s'], label="Optimized", marker='D', lw=2, color='#2ca02c', linestyle=':')

plt.title("Runtime vs. Number of Frequent Itemsets")
plt.xlabel("Frequent itemsets")
plt.ylabel("Runtime (Seconds)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_runtime_vs_itemsets.png"), bbox_inches="tight")
plt.show()

print("\n" + "="*80)
print("BẢNG DỮ LIỆU THỰC NGHIỆM 6 (AVG LENGTH)")
print("="*80)
display(df)

## Chương 3: Ứng dụng thực tế

### 3.1. Ứng dụng 1: Phân tích Giỏ hàng

Để thực hiện phân tích giỏ hàng, tập dữ liệu được sử dụng là **groceries** (dữ liệu bán lẻ thực tế) có định dạng CSV. Mỗi dòng tương ứng với một giao dịch, gồm các item biểu diễn dưới dạng chuỗi (VD: "banana"), ngăn cách nhau bởi dấu phẩy.

Do thuật toán FP-growth được cài đặt sử dụng cấu trúc dữ liệu có thuộc tính nhãn item được biểu diễn bởi số nguyên, kỹ thuật mã hóa các item từ **groceries** sau được áp dụng:
1. Duyệt qua từng dòng của tập dữ liệu, lập chỉ mục ánh xạ một item thành một số nguyên duy nhất thông qua một bảng băm.
2. Chuyển đổi từng dòng giao dịch thành mảng số nguyên theo định dạng của SPMF.
3. Kết quả khai thác luật kết hợp sẽ chuyển đổi cách biểu diễn số nguyên về lại chuỗi tên item.


In [6]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv"
data_dir = "data/market_basket"
os.makedirs(data_dir, exist_ok=True)

raw_path = os.path.join(data_dir, "groceries.csv")
encoded_path = os.path.join(data_dir, "encoded_groceries.txt")
mapping_path = os.path.join(data_dir, "mapping.csv")

if not os.path.exists(raw_path):
    urllib.request.urlretrieve(url, raw_path)

item_to_id = {}
current_id = 1

with open(raw_path, "r", encoding="utf-8") as f, \
     open(encoded_path, "w", encoding="utf-8") as data_out, \
     open(mapping_path, "w", encoding="utf-8") as map_out:
    
    map_out.write("ID,Item\n")
    for line in f:
        items = [i.strip() for i in line.strip().split(",") if i.strip()]
        if not items:
            continue
        
        encoded_transaction = []
        for item in items:
            if item not in item_to_id:
                item_to_id[item] = current_id
                map_out.write(f"{current_id},{item}\n")
                current_id += 1
            encoded_transaction.append(str(item_to_id[item]))
            
        data_out.write(" ".join(encoded_transaction) + "\n")

print(f"Encoded successfully {len(item_to_id)} items into '{data_dir}/'.")

Encoded successfully 169 items into 'data/market_basket/'.


In [8]:
!julia --project=. applications/app1_market_basket.jl --minsup 0.01 --minconf 0.2 --top 10

Mining frequent itemsets (minsup = 0.01, minconf = 0.2)...

--- TOP 10 ASSOCIATION RULES (Based on Lift) ---
Rule (X -> Y)                                      | Support  | Conf(%)  | Lift  
--------------------------------------------------------------------------------
{citrus fruit, other vegetables} => {root vegetables} | 102      | 35.92    | 3.30  
{other vegetables, yogurt} => {whipped/sour cream} | 100      | 23.42    | 3.27  
{other vegetables, tropical fruit} => {root vegetables} | 121      | 34.28    | 3.14  
{beef} => {root vegetables}                        | 171      | 33.14    | 3.04  
{citrus fruit, root vegetables} => {other vegetables} | 102      | 58.62    | 3.03  
{root vegetables, tropical fruit} => {other vegetables} | 121      | 58.45    | 3.02  
{root vegetables} => {other vegetables, whole milk} | 228      | 21.27    | 2.84  
{other vegetables, whole milk} => {root vegetables} | 228      | 30.98    | 2.84  
{butter} => {other vegetables, whole milk}         | 1

In [ ]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

rules_path = "data/results/app1_market_basket/association_rules.csv"
df_rules = pd.read_csv(rules_path)

top10 = df_rules.sort_values(by="Lift", ascending=False).head(10).copy()

conf_col = next((c for c in top10.columns if "conf" in c.lower()), None)
if conf_col and top10[conf_col].max() <= 1.0:
    top10[conf_col] = top10[conf_col] * 100

fmt = {c: "{:.2f}" for c in top10.select_dtypes(include="float").columns}
if conf_col:
    fmt[conf_col] = "{:.2f}%"

pd.set_option("display.max_colwidth", None)
print(f"Total association rules mined: {len(df_rules)}")
display(top10)

Total association rules mined: 234


,Antecedent,Consequent,Support,Confidence_Pct,Lift
0,"{citrus fruit, other vegetables}",{root vegetables},102,35.92,3.30
1,"{other vegetables, yogurt}",{whipped/sour cream},100,23.42,3.27
2,"{other vegetables, tropical fruit}",{root vegetables},121,34.28,3.14
3,{beef},{root vegetables},171,33.14,3.04
4,"{citrus fruit, root vegetables}",{other vegetables},102,58.62,3.03
5,"{root vegetables, tropical fruit}",{other vegetables},121,58.45,3.02
6,{root vegetables},"{other vegetables, whole milk}",228,21.27,2.84
7,"{other vegetables, whole milk}",{root vegetables},228,30.98,2.84
8,{butter},"{other vegetables, whole milk}",113,20.73,2.77
9,"{curd, whole milk}",{yogurt},99,38.52,2.76


Ở trên là kết quả 10 luật kết hợp theo Lift với $minsup = 0.01$ và $minconf = 0.2$.

Có thể thấy, các luật có Lift cao (từ 2.76 đến 3.30) cho thấy hai nhóm mặt hàng $X$ và $Y$ cùng xuất hiện trong một giỏ hàng thường xuyên hơn khoảng $3$ lần so với mức kỳ vọng nếu $X$ và $Y$ độc lập với nhau. Ví dụ, luật $\text{beef} \rightarrow \text{root vegetables}$ có Lift là $3.04$. Nếu hai mặt hàng này không liên quan, ta chỉ kỳ vọng khoảng $56$ giỏ hàng chứa cả hai, nhưng thực tế quan sát được $171$ giỏ. Nói cách khác, khách hàng mua thịt bò có khả năng mua thêm rau củ cao gấp khoảng $3$ lần so với một khách hàng bất kỳ ($33.14\%$ so với $10.9\%$).

Về độ tin cậy, chỉ có $\text{root vegetables, citrus} \rightarrow \text{other vegetables}$ và $\text{tropical fruit} \rightarrow \text{root vegetables}$ được ngưỡng confident $58\%$. Các luật còn lại đạt confident khoảng $20\%$ đến dưới $40\%$. Xét lại ví dụ $\text{beef} \rightarrow \text{root vegetables}$, luật này có confident là $33.14\%$. Như vậy trong số các giỏ hàng có thịt bò, có khoảng 1/3 chúng có thêm rau củ.

Khi quan sát cả Lift và độ tin cậy, ta có một số nhận định như sau:
- Các luật $X \rightarrow Y$ có confident cao nhưng Lift thấp cho thấy do $Y$ vốn dĩ đã phổ biến trong các giỏ hàng của tập dữ liệu nên nhiều giỏ hàng có $X$ cũng có thêm $Y$ nhưng việc khách hàng mua $X$ không ảnh hưởng tới lựa chọn mua $Y$.
- Các luật $X \rightarrow Y$ có Lift cao nhưng confident khiêm tốn cho thấy tồn tại mối quan hệ giữa $X$ và $Y$ trong dữ liệu nhưng $Y$ không thực sự phổ biến để các giỏ hàng có cả $Y$ và $X$ nhiều. Do trường hợp này xuất hiện nhiều trong bảng kết quả, ta nên tận dụng bằng cách đặt các mã giảm giá, quảng cáo mặt hàng rau củ nhắm tới người mua thịt bò.
- Các luật $X \rightarrow Y$ có cả Lift và confident cao nên được khai thác để sắp đặt kệ hàng $Y$ gần $X$, hệ thống tư vấn,... do các luật này dự đoán tốt quyết định mua hàng của người dùng mà không cần chất xúc tác như mã giảm giá, quảng cáo,... ở trên.

Qua các quan sát trên, nhóm đưa ra các quyết định kinh doanh như sau:
- Sắp xếp các kệ hàng theo các luật: đặt kệ hàng các loại rau củ (vegetables và other vegetables) gần với kệ hàng thịt bò và kệ hàng trái cây (citrus và tropical), kệ hàng kem chua (whipped/sour cream) gần với kệ hàng sữa chua, kệ hàng sữa chua gần với kệ hàng sữa nguyên chất và sữa đông.
- Các luật có Lift và confident cao như $\text{root vegetables, citrus} \rightarrow \text{other vegetables}$ và $\text{tropical fruit} \rightarrow \text{root vegetables}$: sử dụng cho hệ thống tư vấn. VD: khi trong giỏ hàng người dùng có rau củ và trái cây chua, gợi ý thêm các loại rau củ khác.
- Các luật có Lift cao và confident thấp (đa số các luật trong bảng): sử dụng các bảng giảm giá khi mua thêm hoặc biển quảng cáo hình ảnh. VD: đặt bảng giảm giá đối với nhóm mặt hàng sau: sữa chua, sữa đông và sữa nguyên chất; ưu đãi giảm giá khi mua thịt bò và rau củ.

### 3.2. Ứng dụng 2: Phân tích dữ liệu sinh học

Tập dữ liệu được sử dụng là **Splice-junction Gene Sequences** từ UCI Machine Learning Repository (giấy phép CC BY 4.0), gồm 3190 đoạn DNA người. Mỗi đoạn dài 60 nucleotide (A, C, G, T), vị trí cần xét nằm giữa nucleotide thứ 30 và 31. Mỗi đoạn được gán một trong ba nhãn:
- **EI** (767 đoạn): ranh giới exon → intron (donor site).
- **IE** (768 đoạn): ranh giới intron → exon (acceptor site).
- **N** (1655 đoạn): không chứa ranh giới, dùng làm nhóm đối chứng.

Do FP-growth xem mỗi giao dịch là một tập hợp không có thứ tự, thông tin vị trí được gắn trực tiếp vào item. Kỹ thuật mã hóa sau được áp dụng:
1. Mỗi đoạn DNA là một giao dịch gồm 60 item dạng (vị trí, nucleotide). VD: item `p31=G` nghĩa là nucleotide ở vị trí 31 là G.
2. Item `pK=B` được mã hóa thành số nguyên $(K-1) \times 4 + i_B$, với $i_B \in \{1, 2, 3, 4\}$ là thứ tự của $B$ trong A, C, G, T, tổng cộng 240 item. Các ký hiệu không xác định (D, N, S, R) bị loại bỏ.
3. Mỗi nhãn được ghi ra một file theo định dạng SPMF, kết quả khai thác được chuyển về tên item qua `splice_mapping.csv`.

In [7]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/splice-junction-gene-sequences/splice.data"
data_dir = "data/splice"
os.makedirs(data_dir, exist_ok=True)
raw_path = os.path.join(data_dir, "splice.data")
if not os.path.exists(raw_path):
    urllib.request.urlretrieve(url, raw_path)

BASES = "ACGT"
def item_id(pos, base):  # pos 1..60, base in ACGT -> id 1..240
    return (pos - 1) * 4 + BASES.index(base) + 1

outs = {c: open(os.path.join(data_dir, f"splice_{c}.txt"), "w", encoding="utf-8") for c in ["EI", "IE", "N"]}
counts = {c: 0 for c in outs}
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        label, name, seq = [s.strip() for s in line.split(",")]
        label = label.upper()
        items = [item_id(i + 1, b) for i, b in enumerate(seq.upper()) if b in BASES]  # remove D/N/S/R
        outs[label].write(" ".join(map(str, items)) + "\n")
        counts[label] += 1
for f in outs.values():
    f.close()

with open(os.path.join(data_dir, "splice_mapping.csv"), "w", encoding="utf-8") as map_out:
    map_out.write("ID,Item\n")
    for pos in range(1, 61):
        for b in BASES:
            map_out.write(f"{item_id(pos, b)},p{pos}={b}\n")
print(f"Encoded sucessfully: {counts}")

Encoded sucessfully: {'EI': 767, 'IE': 768, 'N': 1655}


In [ ]:
!julia --project=. applications/app2_splice.jl --minsup 0.4 --top 10

In [ ]:
dir_path = "data/results/app2_splice"
freq = pd.read_csv(os.path.join(dir_path, "position_freq.csv"))
bases = ["A", "C", "G", "T"]

fig, axes = plt.subplots(3, 1, figsize=(12, 6), dpi=300, sharex=True)
for ax, c in zip(axes, ["EI", "IE", "N"]):
    m = (freq[freq["Class"] == c]
         .pivot(index="Base", columns="Position", values="Freq")
         .reindex(index=bases, columns=range(1, 61)).fillna(0))
    im = ax.imshow(m.values, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    ax.set_yticks(range(4))
    ax.set_yticklabels(bases)
    ax.axvline(29.5, color="red", ls="--", lw=1)  # junction between position 30 and 31
    ax.set_title(f"Class {c}: nucleotide frequency by position")

axes[-1].set_xticks(range(0, 60, 5))
axes[-1].set_xticklabels(range(1, 61, 5))
axes[-1].set_xlabel("Position in DNA sequence (junction between 30 and 31)")
fig.colorbar(im, ax=axes, label="Support")

plt.savefig(os.path.join(dir_path, "position_freq.png"), bbox_inches="tight")
plt.show()

Mỗi lớp được khai thác riêng với $minsup = 0.4$. Để đánh giá mức độ đặc trưng của một tập phổ biến $X$ trong lớp $C \in \{EI, IE\}$ so với nhóm đối chứng, sử dụng độ tăng trưởng (growth rate):

$$GR(X) = \frac{supp_C(X)}{\max\left(supp_N(X),\ 1/|D_N|\right)}$$

trong đó $supp_C(X)$ và $supp_N(X)$ là support tương đối của $X$ trong lớp $C$ và lớp N, $|D_N| = 1655$ là số đoạn DNA của lớp N; cận dưới $1/|D_N|$ tránh chia cho $0$ khi $X$ không xuất hiện trong N.

Kết quả: lớp EI có 63 tập phổ biến, lớp IE có 61 tập phổ biến, trong khi lớp N **không có tập phổ biến nào** ở cùng ngưỡng (item phổ biến nhất của N chỉ đạt $28.9\%$, gần mức ngẫu nhiên $25\%$). Như vậy các mẫu tìm được ở EI và IE gắn liền với điểm nối chứ không phải đặc điểm chung của DNA.

**Lớp EI (exon → intron):**
- $\{p31=G, p32=T\}$ xuất hiện trong $99.0\%$ đoạn EI nhưng chỉ $5.9\%$ đoạn N (xấp xỉ mức ngẫu nhiên $1/16$). Đây chính là quy tắc GT-AG: intron luôn bắt đầu bằng GT.
- Các tập dài hơn quanh điểm nối: $\{p30=G, p31=G, p32=T, p34=A, p35=G\}$ ($47.3\%$ trong EI, không xuất hiện trong N) khớp với trình tự consensus của donor site là MAG|GTRAGT (M là A hoặc C, R là A hoặc G). Vị trí 28 (M) và 33 (R) bị chia support giữa hai nucleotide (vị trí 33: A $49.3\%$, G $46.4\%$) nên ít xuất hiện trong các tập phổ biến dài.
- Tập càng dài thì support càng giảm nhưng growth rate tăng mạnh: $\{p31=G, p32=T\}$ có $GR = 16.7$, còn $\{p31=G, p32=T, p34=A, p35=G\}$ có $GR = 1055.1$. Do GT xuất hiện ngẫu nhiên khá thường xuyên trong DNA, chỉ các mẫu dài mới phân biệt được điểm nối thật.

**Lớp IE (intron → exon):**
- $\{p29=A, p30=G\}$ xuất hiện trong $99.5\%$ đoạn IE: intron luôn kết thúc bằng AG, vế còn lại của quy tắc GT-AG.
- $\{p28=C, p29=A, p30=G\}$ ($78.8\%$, $GR = 34.3$) và item $p31=G$ ($47.5\%$) cho thấy trình tự consensus CAG|G của acceptor site.
- Nhiều tập dạng $\{pK=C \text{ hoặc } T, p29=A, p30=G\}$ với $K$ từ 18 đến 26 đạt support khoảng $42\%$ đến $46\%$ và $GR$ từ 25 đến 31: phía trước AG là vùng giàu C và T (polypyrimidine tract). Vùng này có độ dài và vị trí thay đổi giữa các đoạn nên support bị trải đều ra nhiều vị trí thay vì tập trung như GT và AG.

Ý nghĩa và ứng dụng:
- Chú giải gen: các mẫu trên là "chữ ký" giúp xác định ranh giới exon/intron khi phân tích một chuỗi DNA mới.
- Đặc trưng cho mô hình phân loại: mỗi tập phổ biến có growth rate cao là một đặc trưng nhị phân (đoạn DNA có chứa mẫu hay không) để huấn luyện mô hình phân loại EI/IE/N; các đặc trưng này dễ giải thích hơn đầu vào là chuỗi thô.
- Đột biến làm mất các mẫu này tại điểm nối có thể khiến tế bào cắt nối sai, nên các mẫu cũng là vùng cần ưu tiên kiểm tra khi phân tích đột biến.

Hạn chế của cách mã hóa (vị trí, nucleotide):
- Item gắn với vị trí cố định nên không phát hiện được mẫu bị dịch vị trí giữa các đoạn, như vùng giàu C/T ở trên.
- Vị trí cho phép nhiều nucleotide (M, R) bị chia nhỏ support. Có thể khắc phục bằng cách bổ sung item theo nhóm nucleotide, VD: $pK=R$ (A hoặc G), $pK=Y$ (C hoặc T).